# Data Preprocessing

In [10]:
import pandas as pd

# Path ke file Excel
file_path = "Dari0/Media_Sosial_Mentah.xlsx"

# Membaca file Excel
df = pd.read_excel(file_path)

In [ ]:

# Pastikan nilai di kolom title dan snippet adalah string
df["title"] = df["title"].fillna("").astype(str)
df["snippet"] = df["snippet"].fillna("").astype(str)

# Menyisakan kolom tertentu di DataFrame
df = df[["entity", "timeStart", "medianame", "title", "snippet"]]

# Menggabungkan kolom title dan snippet menjadi kolom baru bernama text
df["text"] = df["title"] + " " + df["snippet"]

# Menampilkan hasil
df.head()

In [ ]:
pip install nltk

In [30]:
df_stopwords = open('data/stopwords_id.txt', "r", encoding="utf-8", errors='replace')
id_stop = df_stopwords.readlines()
df_stopwords.close()
id_stop = [t.strip().lower() for t in id_stop]
id_stop = set(id_stop)

import pandas as pd
import re
from unidecode import unidecode

def cleansing(df):
    # Remove URL
    def remove_url(text):
        url_pattern = re.compile(r"https?://\S+")
        return url_pattern.sub(r"", text)

    # Lowercasing and Unicode normalization
    def normalize_text(text):
        text = text.lower()
        text = unidecode(text)
        return text

    # Tokenization using str.split()
    def tokenize_text(text):
        tokens = text.split()
        tokens = [t for t in tokens if t not in id_stop]  # Remove stopwords
        tokens = [t for t in tokens if len(t) >= 3]  # Remove words < 3 chars
        return tokens

    def remove_url(text):
        url_pattern = re.compile(r"https?://\S+")
        return url_pattern.sub(r"", text)
    
    df["text"] = df["text"].apply(lambda x: remove_url(x))

    # Menghapus semua baris yang memiliki nilai NaN
    df = df.dropna()

    # Lowercasing
    df["lowercase"] = df['text'].apply(lambda x: x.lower())

    # Unicode normalization
    df["lowercase"] = df["lowercase"].apply(lambda x: normalize_text(x))

    # Punctuation removal
    df["lowercase"] = df["lowercase"].apply(lambda x: re.sub(r'[^\w\s]', '', x))

    # Tokenization
    df["token"] = df["lowercase"].apply(lambda x: tokenize_text(x))

    # # Stopwords removal
    df["token"] = df["token"].apply(lambda tokens: [t for t in tokens if t not in id_stop])

    # Removal of words less than 3 characters
    df["token"] = df["token"].apply(lambda x: [t for t in x if len(t) >= 3])

    # Joining tokenized words into cleaned text
    df["cleaned_txt"] = df["token"].apply(lambda tokens: ' '.join(tokens))

    # # Selecting columns 'text', 'label', and 'cleaned_txt'
    df.drop(columns=["lowercase", "token"], inplace=True)

    return df

# Contoh pemanggilan fungsi cleansing
df = cleansing(df)

In [32]:
# Membaca stopwords dari file
df_stopwords = open('data/stopwords_id.txt', "r", encoding="utf-8", errors='replace')
id_stop = df_stopwords.readlines()
df_stopwords.close()
id_stop = [t.strip().lower() for t in id_stop]
id_stop = set(id_stop)

# Menghapus stopwords dari kolom text
def remove_stopwords(text):
    if isinstance(text, str):
        words = text.split()
        filtered_words = [word for word in words if word.lower() not in id_stop]
        return " ".join(filtered_words)
    return text

# Terapkan fungsi ke kolom cleaned_txt
df['cleaned_txt'] = df['cleaned_txt'].apply(remove_stopwords)

# Menyimpan hasil ke file baru (opsional)
output_file = "dari0/cleaned_media_sosial.xlsx"
df.to_excel(output_file, index=False)

In [ ]:
# Path ke file Excel
file_path = "Dari0/word_frequency_sosial.xlsx"

# Membaca file Excel
stopwords = pd.read_excel(file_path)

stopwords.drop(columns=["Word", "Frequency"], inplace=True)

stopwords = stopwords.dropna().values.tolist()

stopwords = [item[0] for item in stopwords]

stopwords

In [ ]:
# Daftar kata yang ingin dihapus
kata_dihapus = ['honda', 'daihatsu', 'suzuki', 'indonesia', 'toyota', 'mercedesbenz', 'hyundai', 
             'brio', 'mobil', '2023', 'wuling', 'com', 'mitsubishi', 'jual', 'mar', 'apr', 'jan', 
             'jul', 'feb', 'jun', 'mei', 'sep', 'okt', 'nov', 'des', 'lowong_kerja', 'camat', 
             'warga', 'terima_kasih', 'terima', 'nyala_march', 'instagram_photos', 'videos_from', 'photos',
             'suzukiindonesia', 'indonesia_suzukiindonesia', 'official', 'januari', 'februari', 
             'maret', 'april', 'mei', 'juni', 'juli', 'agustus', 'september', 'oktober', 'november', 
             'desember', 'nyala_may', 'ayo', 'jakarta', 'lensa', 'berita', 'teman', 
             'kasih', 'link', 'handphone', 'halo', 'whatsapp', 'nyala', 'may', 'march', 'likes', 'comments', 
             'likes_comments', 'nyala', 'instagram', 'twitter', 'facebook', 'tidak', 'dapur pacu', 'moladin', 
             'detikoto', 'otomotif bisnis', 'otomotif kompas', 'otomotifnet', 'ridertua', 'carmudi indonesia', 
             'oto', 'kobayogas', 'car user magz', 'gaikindo', 'blog durable', 'carro blog', 'autopedia', 
             'linkedin', 'youtube', 'tiktok', 'video', 'jawa', 'timur', 'barat', 'selatan', 'utara', 'layan',
             'kirim', 'https', 'info', 'hubung', 'silah', 'tarik', 'beli', 'laku', 'buah', 'harga ',"shvs"]

kata_dihapus = stopwords + kata_dihapus

def hapus_kata(teks):
    pattern = r'\b(?:' + '|'.join(kata_dihapus) + r')\b'
    return re.sub(pattern, '', teks)

# Terapkan fungsi ke kolom 'cleaned_txt'
df['cleaned_txt'] = df['cleaned_txt'].apply(hapus_kata)

# Hapus spasi berlebih yang dihasilkan oleh penghapusan kata
df['cleaned_txt'] = df['cleaned_txt'].str.replace(' +', ' ')

df.head()

In [54]:
df.to_excel("Dari0/cleaned_media_sosial.xlsx", index=False)

In [98]:
import pandas as pd
# Path ke file Excel
file_path = "Dari0/cleaned_media_sosial.xlsx"

# Membaca file Excel
df = pd.read_excel(file_path)

In [ ]:
duplicates = df[df.duplicated()]

if not duplicates.empty:
    print(f"Ditemukan {len(duplicates)} baris duplikat.")
    print(duplicates)
else:
    print("Tidak ada data terduplikat di df_concat.")

In [ ]:
df = df.drop_duplicates()
print("Data duplikat telah dihapus.")

In [ ]:
import pandas as pd
from tqdm import tqdm

path = "text-preprocesing/slang.csv"

slang = pd.read_csv(path)

# Membuat kamus slang -> formal
slang_dict = dict(zip(slang['slang'], slang['formal']))

# Fungsi untuk mengganti slang dengan formal
def replace_slang(text):
    if not isinstance(text, str):  # Cek apakah text adalah string
        return text  # Jika bukan string (misalnya NaN), kembalikan tanpa perubahan
    words = text.split()  # Memecah kalimat menjadi kata-kata
    new_words = [slang_dict.get(word, word) for word in words]  # Ganti dengan formal jika ada
    return ' '.join(new_words)

# Menggunakan tqdm untuk menambahkan progress bar
tqdm.pandas()

# Misalnya, DataFrame Anda memiliki kolom 'cleaned_txt'
df['cleaned_txt'] = df['cleaned_txt'].progress_apply(replace_slang)

# Tampilkan DataFrame setelah penggantian
df['cleaned_txt']

# Semi-supervised LDA

In [ ]:
import warnings; warnings.simplefilter('ignore')
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import pyLDAvis, pyLDAvis.lda_model; pyLDAvis.enable_notebook()
import gensim, numpy as np, math
from unidecode import unidecode
from html import unescape
from textblob import TextBlob
from tqdm import tqdm
from lda import guidedlda
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA

sns.set(style="ticks", color_codes=True)
random_state = 0

"Done"

In [106]:
def get_umass_score(dt_matrix, i, j):
    zo_matrix = (dt_matrix > 0).astype(int)
    col_i, col_j = zo_matrix[:, i], zo_matrix[:, j]
    col_ij = col_i + col_j
    col_ij = (col_ij == 2).astype(int)    
    Di, Dij = col_i.sum(), col_ij.sum()    
    return math.log((Dij + 1) / Di)

def get_topic_coherence(dt_matrix, topic, n_top_words):
    indexed_topic = zip(topic, range(0, len(topic)))
    topic_top = sorted(indexed_topic, key=lambda x: 1 - x[0])[0:n_top_words]
    coherence = 0
    for j_index in range(0, len(topic_top)):
        for i_index in range(0, j_index - 1):
            i = topic_top[i_index][1]
            j = topic_top[j_index][1]
            coherence += get_umass_score(dt_matrix, i, j)
    return coherence

def get_average_topic_coherence(dt_matrix, topics, n_top_words):
    total_coherence = 0
    for i in range(0, len(topics)):
        total_coherence += get_topic_coherence(dt_matrix, topics[i], n_top_words)
    return total_coherence / len(topics)

In [ ]:
try:
    fData = 'data_aspek_otomotif_new.csv'
    dfL = pd.read_csv(fData)
except Exception as err_:  
    print(err_)
    print("Wajib menjalankan 'Sentimen_Analisis_Data_PreProcessing.ipynb' terlebih dahulu.")

print(df.shape)
print(df.info())

In [108]:
# # List nama-nama media yang ingin diambil
# media_list = ['Dapur Pacu', 'Moladin', 'DetikOto', 'Otomotif Bisnis', 'Otomotif Kompas',
#               'Otomotifnet', 'Ridertua', 'Carmudi Indonesia', 'Oto', 'Kobayogas',
#               'Car User MAGZ', 'Gaikindo', 'Blog Durable', 'Carro Blog', 'Autopedia']

# # Seleksi baris berdasarkan nilai medianame dalam media_list
# df = df[df['medianame'].isin(media_list)]

In [109]:
# Hapus kolom 'Aspek'
df = df.dropna(subset = ['cleaned_txt'])
df = df.reset_index(drop=True)

In [ ]:
seed_topic_list = {c.lower().strip():[] for c in dfL.columns}
for i, d in dfL.iterrows():
    for c in dfL.columns:
        if not pd.isna(d[c]) and not pd.isnull(d[c]):
            seed_topic_list[c.lower().strip()].append(unescape(unidecode(d[c])).lower().strip())
print(str(seed_topic_list)[:150])

nTopics = len(seed_topic_list)
nTopics

In [ ]:
vsm = CountVectorizer(binary = False, ngram_range=(1, 3), min_df=1, max_df=1.0)

tf = vsm.fit_transform(df['cleaned_txt'])
tf_terms = vsm.get_feature_names_out()
word2id = dict((v, idx) for idx, v in enumerate(tf_terms))

print(tf.shape, type(tf))

In [ ]:
from lda import guidedlda

"Done"

In [ ]:
allWords = vsm.get_feature_names_out().tolist()
word2id = {kata:i for i, kata in enumerate(vsm.get_feature_names_out())}
bidang = list(seed_topic_list.keys())
Biaya = [word2id[kata] for kata in allWords if kata in seed_topic_list['biaya']]
Keselamatan = [word2id[kata] for kata in allWords if kata in seed_topic_list['keselamatan']]
Lingkungan = [word2id[kata] for kata in allWords if kata in seed_topic_list['lingkungan']]
Fitur = [word2id[kata] for kata in allWords if kata in seed_topic_list['fitur']]
Reliabilitas = [word2id[kata] for kata in allWords if kata in seed_topic_list['reliabilitas']]

seed_list = [Biaya, Keselamatan, Lingkungan, Fitur, Reliabilitas]
print("Banyak kata di setiap bidang: ", [len(bidang) for bidang in seed_list])
print("Contoh 5 nilai index kata di bidang Biaya = ", Biaya[:5])

In [ ]:
seed_topics = {}
for t_id, st in enumerate(seed_list):
    for idx in st:
        seed_topics[idx] = t_id

model_guided = guidedlda.GuidedLDA(n_topics=nTopics, n_iter=300, random_state=random_state, refresh=30)
model_guided.fit(tf, seed_topics=seed_topics, seed_confidence=0.95)

In [150]:
print("Log Likelihood Guided LDA = ", max(model_guided.loglikelihoods_))a

In [151]:
# The lower UMass coherence - the better.
# https://stackoverflow.com/questions/69730428/how-do-i-find-coherence-score-for-lsa-and-lda-for-sklearn-models
n_top_words = 30
topics = model_guided.transform(tf)
coh_score = get_average_topic_coherence(tf, topics, n_top_words)
print("Coherence Score Semi-Supervised LDA = ", coh_score)

# Maping Aspect to Every Seed in Semi-Supervised LDA Topic Modelling

In [ ]:
print("Kata-kata di VSM ada sebanyak = ", len(vsm.get_feature_names_out()))
print("Ukuran matrix Topik X Kata = ", model_guided.components_.shape)
print("Menghitung probabilitas setiap topik ke Seed: ... ")
nTopics = model_guided.components_.shape[0]
nKata = model_guided.components_.shape[1]
id2word = {i:kata for i, kata in enumerate(vsm.get_feature_names_out())}
topicSeed = {bidang:[0.0]*nTopics for bidang in seed_topic_list.keys()}
for bidang in topicSeed.keys():
    for topic_ in range(nTopics):
        for j in range(nKata):
            if id2word[j] in seed_topic_list[bidang]:
                topicSeed[bidang][topic_] += model_guided.components_[topic_, j]
    range_ = np.sum(topicSeed[bidang])
    if range_>0.0:
        topicSeed[bidang] = [nilai/range_ for nilai in topicSeed[bidang]]
topicSeed = pd.DataFrame(topicSeed)
topicSeed.head(nTopics)

In [ ]:
# melihat topik yang dominan pada tiap dokumen 
# Create Document - Topic Matrix
lda_output = model_guided.transform(tf)

# column names
topicnames = ["Topic" + str(i) for i in range(len(model_guided.components_))]

# index names
docnames = ["Doc" + str(i) for i in range(len(df['cleaned_txt']))]

# Make the pandas dataframe
df_document_topic = pd.DataFrame(np.round(lda_output, 10), columns=topicnames, index=docnames)

# Get dominant topic for each document
dominant_topic = np.argmax(df_document_topic.values, axis=1)
df_document_topic['dominant_topic'] = dominant_topic

# Styling
def color_green(val):
    color = 'green' if val > .1 else 'black'
    return 'color: {col}'.format(col=color)

def make_bold(val):
    weight = 700 if val > .1 else 400
    return 'font-weight: {weight}'.format(weight=weight)

# Apply Style
df_document_topics = df_document_topic.head(10).style.applymap(color_green).applymap(make_bold)
df_document_topics

In [ ]:
# memaknai masing-masing topik dengan data frame
# Show top n keywords for each topic
def show_topics(vectorizer=vsm, lda_model=model_guided, n_words=10):
    keywords = np.array(vectorizer.get_feature_names_out())
    topic_keywords = []
    for topic_weights in lda_model.components_:
        top_keyword_locs = (-topic_weights).argsort()[:n_words]
        topic_keywords.append(keywords.take(top_keyword_locs))
    return topic_keywords


topic_keywords = show_topics(vectorizer=vsm, lda_model=model_guided, n_words=10)        

# Topic - Keywords Dataframe
df_topic_keywords = pd.DataFrame(topic_keywords)
df_topic_keywords.columns = ['Word '+str(i) for i in range(df_topic_keywords.shape[1])]
df_topic_keywords.index = ['Topic '+str(i) for i in range(df_topic_keywords.shape[0])]
df_topic_keywords

In [ ]:
# melihat distribusi topik di seluruh dokumen 
df_topic_distribution = df_document_topic['dominant_topic'].value_counts().reset_index(name="Num Documents")
df_topic_distribution.columns = ['Topic Num', 'Num Documents']
df_topic_distribution.sort_values(by=['Topic Num'], inplace=True)
p = sns.countplot(x=df_document_topic['dominant_topic'])
df_topic_distribution

In [126]:
pyLDAvis.lda_model.prepare(model_guided, tf, vsm)
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Replace `prepared_data` with your prepared LDAvis data
vis_data = pyLDAvis.lda_model.prepare(model_guided, tf, vsm)

# Save the visualization to an HTML file
pyLDAvis.save_html(vis_data, 'lda_visualization.html')

In [ ]:
df_document_topic.info()    

In [128]:
# Membuat mapping dari angka ke kata-kata yang sesuai
topic_mapping = {
    0: 'biaya',
    1: 'keselamatan',
    2: 'lingkungan',
    3: 'fitur',
    4: 'reliabilitas',
    5: "lainnya"
}

df_document_topic['dominant_topic']

# Tampilkan hasilnya
df_document_topic1 = df_document_topic['dominant_topic'].reset_index(drop=True)

In [ ]:
 # Menggunakan map untuk menerapkan mapping ke kolom 'dominant_topic'
df_document_topic1 = df_document_topic1.to_frame()
df_document_topic1['dominant_topic'] = df_document_topic1['dominant_topic'].map(topic_mapping)
df_document_topic1

In [ ]:
# Mengganti nilai dalam kolom 'dominant_topic' dengan kata-kata yang sesuai
# Menggabungkan df_label dengan df berdasarkan indeksnya
df_concat = pd.concat([df, df_document_topic1], axis=1)

# Menampilkan hasil gabungan
df_concat

In [ ]:
duplicates = df_concat[df_concat.duplicated()]

if not duplicates.empty:
    print(f"Ditemukan {len(duplicates)} baris duplikat.")
    print(duplicates)
else:
    print("Tidak ada data terduplikat di df_concat.")

In [ ]:
df_concat = df_concat.drop_duplicates()
print("Data duplikat telah dihapus.")


In [ ]:
# Menyimpan DataFrame df_concat ke file Excel
df_concat.to_excel("Dari0/media_sosial_rapih.xlsx", index=False)

print("DataFrame berhasil disimpan ke file 'df_concat_output.xlsx'")

In [ ]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import ngrams
from collections import Counter

# Function to preprocess text
def preprocess_text(text):
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and punctuation
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token.isalnum() and token not in stop_words]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return tokens

# Function to extract n-grams
def extract_ngrams(tokens, n):
    ngrams_list = list(ngrams(tokens, n))
    return [' '.join(ngram) for ngram in ngrams_list]

# Function to analyze text data
def analyze_text_data(df, title):
    # Join all text data into a single string
    text_data = ' '.join(df['cleaned_txt'].astype(str))
    
    # Preprocess text
    tokens = preprocess_text(text_data)
    
    # Calculate total number of tokens
    total_tokens = len(tokens)
    
    # Extract unigrams, bi-grams, and tri-grams
    unigrams = tokens
    bigrams = extract_ngrams(tokens, 2)
    trigrams = extract_ngrams(tokens, 3)
    
    # Calculate total number of bigrams and trigrams
    total_bigrams = len(bigrams)
    total_trigrams = len(trigrams)
    
    # Calculate frequency of unigrams, bi-grams, and tri-grams
    unigrams_freq = Counter(unigrams)
    bigrams_freq = Counter(bigrams)
    trigrams_freq = Counter(trigrams)
    
    # Calculate percentage frequencies
    unigrams_percentage_freq = {word: (count / total_tokens) * 100 for word, count in unigrams_freq.items()}
    bigrams_percentage_freq = {bigram: (count / total_bigrams) * 100 for bigram, count in bigrams_freq.items()}
    trigrams_percentage_freq = {trigram: (count / total_trigrams) * 100 for trigram, count in trigrams_freq.items()}
    
    # Print results
    print(f"### Analysis for: {title} ###")
    
    print("\nTop 10 most frequent words (Unigrams):")
    for word, freq in unigrams_freq.most_common(1000):
        print(f"{word}: {unigrams_percentage_freq[word]:.2f}%")
    
#     print("\nTop 20 most frequent bi-grams:")
#     for bigram, freq in bigrams_freq.most_common(20):
#         print(f"{bigram}: {bigrams_percentage_freq[bigram]:.2f}%")
    
#     print("\nTop 20 most frequent tri-grams:")
#     for trigram, freq in trigrams_freq.most_common(20):
#         print(f"{trigram}: {trigrams_percentage_freq[trigram]:.2f}%")
    
# Analyze df_concat
analyze_text_data(df, "Media Massa")

In [139]:
# Mengganti nilai dalam kolom 'dominant_topic' dengan kata-kata yang sesuai
# Menggabungkan df_label dengan df berdasarkan indeksnya
# Membuat mapping dari angka ke kata-kata yang sesuai
topic_mapping = {
    0: 'biaya',
    1: 'keselamatan',
    2: 'lingkungan',
    3: 'fitur',
    4: 'reliabilitas',
    5: 'lainnya' 
}

df_document_topic['dominant_topic']

# Tampilkan hasilnya
df_document_topic1 = df_document_topic['dominant_topic'].reset_index(drop=True)
df_document_topic1 = df_document_topic1.to_frame()
df_document_topic1['dominant_topic'] = df_document_topic1['dominant_topic'].map(topic_mapping)
df_concat = pd.concat([df, df_document_topic1], axis=1)
df_lingkungan = df_concat[df_concat['dominant_topic'].isin(['lingkungan'])]
df_keselamatan = df_concat[df_concat['dominant_topic'].isin(['keselamatan'])]
df_biaya = df_concat[df_concat['dominant_topic'].isin(['biaya'])]
df_reliabilitas = df_concat[df_concat['dominant_topic'].isin(['reliabilitas'])]
df_fitur = df_concat[df_concat['dominant_topic'].isin(['fitur'])]
df_lainnya = df_concat[df_concat['dominant_topic'].isin(['lainnya'])]

# Menampilkan hasil gabungan
# df_concat = df_concat.drop(df.columns[11], axis=1)
# df_concat

In [140]:
df_concat = df_concat.rename(columns=lambda x: x + '_dup' if x in df_concat.columns.duplicated() else x)

In [ ]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import ngrams
from collections import Counter

# Function to preprocess text
def preprocess_text(text):
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and punctuation
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token.isalnum() and token not in stop_words]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return tokens

# Function to extract n-grams
def extract_ngrams(tokens, n):
    ngrams_list = list(ngrams(tokens, n))
    return [' '.join(ngram) for ngram in ngrams_list]

# Function to analyze text data
def analyze_text_data(df, title):
    # Join all text data into a single string
    text_data = ' '.join(df['cleaned_txt'].astype(str))
    
    # Preprocess text
    tokens = preprocess_text(text_data)
    
    # Calculate total number of tokens
    total_tokens = len(tokens)
    
    # Extract unigrams, bi-grams, and tri-grams
    unigrams = tokens
    bigrams = extract_ngrams(tokens, 2)
    trigrams = extract_ngrams(tokens, 3)
    
    # Calculate total number of bigrams and trigrams
    total_bigrams = len(bigrams)
    total_trigrams = len(trigrams)
    
    # Calculate frequency of unigrams, bi-grams, and tri-grams
    unigrams_freq = Counter(unigrams)
    bigrams_freq = Counter(bigrams)
    trigrams_freq = Counter(trigrams)
    
    # Calculate percentage frequencies
    unigrams_percentage_freq = {word: (count / total_tokens) * 100 for word, count in unigrams_freq.items()}
    bigrams_percentage_freq = {bigram: (count / total_bigrams) * 100 for bigram, count in bigrams_freq.items()}
    trigrams_percentage_freq = {trigram: (count / total_trigrams) * 100 for trigram, count in trigrams_freq.items()}
    
    # Create a DataFrame for unigrams
    unigrams_df = pd.DataFrame(list(unigrams_freq.items()), columns=['Unigram', 'Frequency'])
    unigrams_df['Percentage'] = unigrams_df['Unigram'].map(unigrams_percentage_freq)
    
    # Save the DataFrame to an Excel file
    # unigrams_df.to_excel(f'{title}_unigrams.xlsx', index=False)
    
    # Print a
    print(f"### Analysis for: {title} ###")
    
    print("\nTop 10 most frequent words (Unigrams):")
    for word, freq in unigrams_freq.most_common(30):
        print(f"{word}: {unigrams_percentage_freq[word]:.2f}%")
    
# Assuming df is your dataframe
# Analyze df_concat and save unigrams to Excel
# analyze_text_data(df, "Media_Massa")
# Analisis setiap subset data
analyze_text_data(df_lingkungan, "lingkungan")
analyze_text_data(df_keselamatan, "keselamatan")
analyze_text_data(df_biaya, "biaya")
analyze_text_data(df_reliabilitas, "reliabilitas")
analyze_text_data(df_fitur, "fitur")
analyze_text_data(df_lainnya, "lainnya")


# Finish!

In [144]:
df_lingkungan = df_concat[df_concat['dominant_topic'].isin(['lingkungan'])]
df_keselamatan = df_concat[df_concat['dominant_topic'].isin(['keselamatan'])]
df_biaya = df_concat[df_concat['dominant_topic'].isin(['biaya'])]
df_reliabilitas = df_concat[df_concat['dominant_topic'].isin(['reliabilitas'])]
df_fitur = df_concat[df_concat['dominant_topic'].isin(['fitur'])]
df_lainnya = df_concat[df_concat['dominant_topic'].isin(['lainnya'])]

In [145]:
import gensim
from gensim.models import Phrases
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.models.coherencemodel import CoherenceModel

def preprocess_and_lda(df, text_column='cleaned_txt', num_topics=1):
    # Function to preprocess sentences into words
    def sent_to_words(sentences):
        for sentence in sentences:
            yield gensim.utils.simple_preprocess(str(sentence), deacc=True)

    # Preprocess your text data
    x = df[text_column].values.tolist()
    docs = list(sent_to_words(x))

    # Build bigram and trigram models
    bigram = Phrases(docs, min_count=10)
    trigram = Phrases(bigram[docs])

    # Apply the models to your documents
    for idx in range(len(docs)):
        for token in bigram[docs[idx]]:
            if '_' in token:
                docs[idx].append(token)
        for token in trigram[docs[idx]]:
            if '_' in token:
                docs[idx].append(token)

    # Create a dictionary representation of the documents
    dictionary = Dictionary(docs)
    dictionary.filter_extremes(no_below=10, no_above=0.2)

    # Create corpus using the dictionary
    corpus = [dictionary.doc2bow(doc) for doc in docs]

    # Set LDA model parameters
    chunksize = 500 
    passes = 20 
    iterations = 400
    eval_every = 1  

    # Initialize LDA model
    lda_model = LdaModel(corpus=corpus, id2word=dictionary, chunksize=chunksize, \
                         alpha='auto', eta='auto', \
                         iterations=iterations, num_topics=num_topics, \
                         passes=passes, eval_every=eval_every)

    # Compute coherence score
    coherence_model_lda = CoherenceModel(model=lda_model, texts=docs, dictionary=dictionary, coherence='c_v')
    coherence_lda = coherence_model_lda.get_coherence()

    # Print the topics found by the LDA model
    print(lda_model.print_topics())

    # Print the coherence score
    print('Coherence Score: ', coherence_lda)

    # Return the topics model and coherence score
    return lda_model, coherence_lda

# # Example usage:
# # Assume df is your DataFrame with a column 'cleaned_txt'
# lda_model, coherence_score = preprocess_and_lda(df_concat)

In [ ]:
# Definisikan list untuk menyimpan hasil LDA dan koherensi dari setiap topik
results = []
# Iterasi untuk setiap DataFrame
for name, df in [('lingkungan', df_lingkungan), ('keselamatan', df_keselamatan), 
                 ('biaya', df_biaya), ('reliabilitas', df_reliabilitas), 
                 ('fitur', df_fitur), ('lainnya', df_lainnya)]:
    # Melakukan LDA dan preprocessing untuk setiap df
    lda_model, coherence_score = preprocess_and_lda(df)
    
    # Menyimpan hasil ke dalam list
    results.append({
        'topic': name,
        'lda_model': lda_model,
        'coherence_score': coherence_score
    })

# Contoh akses hasil
for result in results:
    print(f"Topic: {result['topic']}, Coherence Score: {result['coherence_score']}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Menghitung frekuensi setiap aspek
aspek_counts = df_concat['dominant_topic'].value_counts()

# Membuat pie chart
plt.figure(figsize=(8, 8))
plt.pie(aspek_counts, labels=aspek_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribusi Topik Hasil LDA Pada Gabungan Media')
plt.show()

In [ ]:
df_concat